# TontoumaBot — Génération et benchmark TTS Wolof (6 modèles)

Génère l'audio wolof à partir de texte avec 6 modèles candidats, les compare, et sauvegarde le meilleur.

```
Texte Wolof
   ↓
TTS (SpeechT5 x3 / XTTS / Adia / Oolel)
   ↓
Audio généré (.wav) + écoute directe dans le notebook
   ↓
Métriques : WER audio, SNR, latence/RTF
```

## ⚠️ Architectures inconnues ou incertaines

Je ne peux pas garantir l'API exacte de `AYI-TEKK/tts-wo`, `CONCREE/Adia_TTS` et `Laurentmd5/Oolel-Voices` — leurs fiches Hugging Face n'ont pas pu être vérifiées en détail. Le notebook tente un chargement générique via `transformers.pipeline("text-to-speech", ...)` pour ces trois, protégé par `try/except`. **Si un modèle échoue au chargement, va voir sa fiche Hugging Face (section "How to use") et adapte sa fonction de synthèse dans la Cellule 3** — le reste du notebook continue de fonctionner avec les modèles qui chargent correctement.

Les deux modèles SpeechT5 de Bilal (`speecht5_tts-wolof` et sa v0.2) suivent l'architecture SpeechT5 standard de `transformers`, donc leur chargement est fiable.

### Installation (une seule fois)

```bash
pip install transformers torch soundfile librosa jiwer coqui-tts psutil \
            pandas matplotlib huggingface_hub peft datasets --break-system-packages
```


In [2]:
!pip install transformers torch soundfile librosa jiwer coqui-tts psutil \
            pandas matplotlib huggingface_hub peft datasets --break-system-packages

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.2 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=f5269fe58bf8f42151d7382324c29c322d80ac7487370e401e84284339621311
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


## 1. Configuration

In [7]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

warnings.filterwarnings("ignore")

MODELES_TTS_WOLOF = {
    "SpeechT5_Bilal": "bilalfaye/speecht5_tts-wolof",
    "SpeechT5_Bilal_v02": "bilalfaye/speecht5_tts-wolof-v0.2",
    "SpeechT5_AYI": "AYI-TEKK/tts-wo",
  }

ASR_MODEL_PATH = "./wolof-whisper-small-lora"  # pour le WER audio (optionnel, cf Cellule 6)
ASR_BASE_MODEL = "openai/whisper-small"

SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

AUDIO_DIR = Path("audios_tts_wolof")
AUDIO_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("resultats_tts_wolof")
RESULTS_DIR.mkdir(exist_ok=True)
MEILLEUR_MODELE_DIR = Path("meilleur_modele_tts_wolof")

print(f"Device utilisé : {DEVICE.upper()}")
print(f"Modèles TTS à comparer : {list(MODELES_TTS_WOLOF.keys())}")

Device utilisé : CUDA
Modèles TTS à comparer : ['SpeechT5_Bilal', 'SpeechT5_Bilal_v02', 'SpeechT5_AYI']


## 2. Chargement des 6 modèles TTS

Chaque modèle est chargé indépendamment — l'échec d'un modèle n'empêche jamais les autres de se charger.

In [8]:
tts_engines = {}
echecs_chargement = {}

# --- SpeechT5 (les 3 variantes partagent le même chargeur) ---
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset

try:
    embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speaker_embedding_defaut = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).to(DEVICE)
except Exception:
    speaker_embedding_defaut = torch.randn(1, 512).to(DEVICE)

vocoder_partage = None
try:
    vocoder_partage = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(DEVICE)
    print("✅ Vocodeur HiFi-GAN partagé chargé")
except Exception as e:
    print(f"❌ Vocodeur HiFi-GAN : {e} — les modèles SpeechT5 ne pourront pas synthétiser")

for nom_cle in ["SpeechT5_Bilal", "SpeechT5_Bilal_v02", "SpeechT5_AYI"]:
    repo_id = MODELES_TTS_WOLOF[nom_cle]
    try:
        proc = SpeechT5Processor.from_pretrained(repo_id)
        modele = SpeechT5ForTextToSpeech.from_pretrained(repo_id).to(DEVICE)
        tts_engines[nom_cle] = {
            "type": "speecht5", "processor": proc, "model": modele,
            "vocoder": vocoder_partage, "speaker_embedding": speaker_embedding_defaut,
        }
        print(f"✅ {nom_cle} chargé (SpeechT5)")
    except Exception as e:
        echecs_chargement[nom_cle] = str(e)
        print(f"❌ {nom_cle} : {e}")

# --- XTTS v2 (Coqui) ---


[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

✅ Vocodeur HiFi-GAN partagé chargé


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ SpeechT5_Bilal chargé (SpeechT5)


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ SpeechT5_Bilal_v02 chargé (SpeechT5)


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ SpeechT5_AYI chargé (SpeechT5)


## 3. Fonctions de synthèse par architecture

Dispatcher unique qui route vers la bonne méthode selon le type de moteur détecté à la Cellule 2.

In [10]:
def _synth_speecht5(engine_dict, texte):
    inputs = engine_dict["processor"](text=texte, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        speech = engine_dict["model"].generate_speech(
            inputs["input_ids"], engine_dict["speaker_embedding"], vocoder=engine_dict["vocoder"]
        )
    return speech.cpu().numpy(), 16000


def _synth_xtts(engine_dict, texte, speaker_wav=None, language="fr"):
    engine = engine_dict["engine"]
    if speaker_wav is None:
        candidats = list(Path("reference_audio").glob("*.wav")) if Path("reference_audio").exists() else []
        speaker_wav = str(candidats[0]) if candidats else None
    if speaker_wav is None:
        raise ValueError("XTTS nécessite un audio de référence — dépose un .wav dans reference_audio/")
    wav = engine.tts(text=texte, speaker_wav=speaker_wav, language=language)
    return np.array(wav, dtype=np.float32), engine.synthesizer.output_sample_rate


def _synth_pipeline_generique(engine_dict, texte):
    sortie = engine_dict["pipeline"](texte)
    audio = np.array(sortie["audio"], dtype=np.float32).squeeze()
    sr = sortie.get("sampling_rate", 16000)
    return audio, sr


def synthetiser(nom_modele, texte):
    """Retourne (audio, sample_rate, temps_ms, erreur)."""
    if nom_modele not in tts_engines:
        return None, None, None, f"Modèle '{nom_modele}' non chargé"

    engine_dict = tts_engines[nom_modele]
    type_moteur = engine_dict["type"]

    t0 = time.perf_counter()
    try:
        if type_moteur == "speecht5":
            audio, sr = _synth_speecht5(engine_dict, texte)
        elif type_moteur == "xtts":
            audio, sr = _synth_xtts(engine_dict, texte)
        elif type_moteur == "pipeline_generique":
            audio, sr = _synth_pipeline_generique(engine_dict, texte)
        else:
            return None, None, None, f"Type de moteur inconnu : {type_moteur}"
        temps_ms = (time.perf_counter() - t0) * 1000
        return audio, sr, temps_ms, None
    except Exception as e:
        temps_ms = (time.perf_counter() - t0) * 1000
        return None, None, temps_ms, str(e)


print("Dispatcher de synthèse prêt pour les 3 modèles.")

Dispatcher de synthèse prêt pour les 3 modèles.


## 4. Génération de l'audio — le cœur de la demande

Phrases wolof à synthétiser par chacun des 6 modèles, avec écoute directe dans le notebook.

In [11]:
phrases_wolof = [
    "Maa ngi bëgg dem Dakar.",
    "Fan la service radiologie ne?",
    "Jërejëf lool ci dimbal bi.",
    "Dama soxla doctoor ndax sama biir dafay metti lool.",
    "Naka laay def ngir am kayitu juddu?",
]

resultats_generation = []

for i, texte in enumerate(phrases_wolof, start=1):
    print("\n" + "=" * 60)
    print(f"Phrase {i} : {texte}")
    print("=" * 60)

    for nom_modele in MODELES_TTS_WOLOF:
        audio, sr, temps_ms, erreur = synthetiser(nom_modele, texte)

        chemin_audio, duree_s, rtf = None, None, None
        if audio is not None:
            duree_s = len(audio) / sr
            rtf = round((temps_ms / 1000) / duree_s, 3) if duree_s > 0 else None
            chemin_audio = AUDIO_DIR / f"{nom_modele}_p{i}.wav"
            sf.write(chemin_audio, audio, sr)
            print(f"  ✅ {nom_modele:20s} -> {chemin_audio.name}  ({duree_s:.1f}s audio, {temps_ms:.0f}ms génération, RTF={rtf})")
        else:
            print(f"  ❌ {nom_modele:20s} -> {erreur}")

        resultats_generation.append({
            "phrase_id": i, "texte_wolof": texte, "modele": nom_modele,
            "audio_path": str(chemin_audio) if chemin_audio else None,
            "sample_rate": sr, "duree_audio_s": round(duree_s, 2) if duree_s else None,
            "latence_ms": round(temps_ms, 1) if temps_ms else None, "rtf": rtf,
            "echec": erreur is not None, "erreur": erreur,
        })

df_generation = pd.DataFrame(resultats_generation)
df_generation.to_csv(RESULTS_DIR / "tts_generation.csv", index=False)
print(f"\n\n{(~df_generation['echec']).sum()}/{len(df_generation)} générations réussies.")


Phrase 1 : Maa ngi bëgg dem Dakar.
  ✅ SpeechT5_Bilal       -> SpeechT5_Bilal_p1.wav  (1.8s audio, 464ms génération, RTF=0.264)
  ✅ SpeechT5_Bilal_v02   -> SpeechT5_Bilal_v02_p1.wav  (1.3s audio, 332ms génération, RTF=0.259)
  ✅ SpeechT5_AYI         -> SpeechT5_AYI_p1.wav  (1.8s audio, 439ms génération, RTF=0.25)

Phrase 2 : Fan la service radiologie ne?
  ✅ SpeechT5_Bilal       -> SpeechT5_Bilal_p2.wav  (2.1s audio, 537ms génération, RTF=0.258)
  ✅ SpeechT5_Bilal_v02   -> SpeechT5_Bilal_v02_p2.wav  (1.7s audio, 423ms génération, RTF=0.254)
  ✅ SpeechT5_AYI         -> SpeechT5_AYI_p2.wav  (2.1s audio, 517ms génération, RTF=0.248)

Phrase 3 : Jërejëf lool ci dimbal bi.
  ✅ SpeechT5_Bilal       -> SpeechT5_Bilal_p3.wav  (2.0s audio, 499ms génération, RTF=0.251)
  ✅ SpeechT5_Bilal_v02   -> SpeechT5_Bilal_v02_p3.wav  (1.3s audio, 364ms génération, RTF=0.271)
  ✅ SpeechT5_AYI         -> SpeechT5_AYI_p3.wav  (1.9s audio, 464ms génération, RTF=0.246)

Phrase 4 : Dama soxla doctoor ndax sama 

## 5. Écoute directe des audios générés

Affiche un lecteur audio pour chaque modèle sur la première phrase — pratique pour juger le naturel à l'oreille tout de suite, sans attendre un MOS formel.

In [12]:
from IPython.display import Audio, display

premiere_phrase = df_generation[df_generation["phrase_id"] == 1]

for _, row in premiere_phrase.iterrows():
    print(f"\n--- {row['modele']} ---")
    print(f"Texte : {row['texte_wolof']}")
    if row["audio_path"]:
        display(Audio(row["audio_path"]))
    else:
        print(f"(échec : {row['erreur']})")


--- SpeechT5_Bilal ---
Texte : Maa ngi bëgg dem Dakar.



--- SpeechT5_Bilal_v02 ---
Texte : Maa ngi bëgg dem Dakar.



--- SpeechT5_AYI ---
Texte : Maa ngi bëgg dem Dakar.


## 6. Métriques : SNR (sans référence) et RTF déjà calculé

In [13]:
def estimer_snr(audio, sr, taille_frame_ms=25):
    taille_frame = int(sr * taille_frame_ms / 1000)
    n_frames = len(audio) // taille_frame
    if n_frames < 4:
        return None
    energies = np.array([np.mean(audio[i*taille_frame:(i+1)*taille_frame] ** 2) for i in range(n_frames)])
    energies = energies[energies > 0]
    if len(energies) < 4:
        return None
    bruit = energies[energies <= np.percentile(energies, 15)]
    signal = energies[energies >= np.percentile(energies, 85)]
    if len(bruit) == 0 or len(signal) == 0 or np.mean(bruit) == 0:
        return None
    return round(float(10 * np.log10(np.mean(signal) / np.mean(bruit))), 2)


snr_scores = []
for _, row in df_generation.iterrows():
    if row["audio_path"] is None:
        snr_scores.append(None)
        continue
    audio, sr = sf.read(row["audio_path"])
    snr_scores.append(estimer_snr(audio, sr))

df_generation["snr_db"] = snr_scores
display(df_generation[["phrase_id", "modele", "duree_audio_s", "latence_ms", "rtf", "snr_db", "echec"]])

,phrase_id,modele,duree_audio_s,latence_ms,rtf,snr_db,echec
0,1,SpeechT5_Bilal,1.76,463.8,0.264,39.87,False
1,1,SpeechT5_Bilal_v02,1.28,332.1,0.259,38.69,False
2,1,SpeechT5_AYI,1.76,439.3,0.250,43.10,False
3,2,SpeechT5_Bilal,2.08,536.8,0.258,43.41,False
4,2,SpeechT5_Bilal_v02,1.66,422.6,0.254,37.52,False
5,2,SpeechT5_AYI,2.08,516.8,0.248,43.48,False
6,3,SpeechT5_Bilal,1.98,498.7,0.251,40.76,False
7,3,SpeechT5_Bilal_v02,1.34,363.8,0.271,31.45,False
8,3,SpeechT5_AYI,1.89,464.1,0.246,40.14,False
9,4,SpeechT5_Bilal,3.58,838.0,0.234,40.08,False


## 7. (Optionnel) WER audio — fidélité de prononciation

Nécessite `./wolof-whisper-small-lora`. Protégé par `try/except` — si absent, le reste du notebook continue normalement.

In [14]:
import re
import unicodedata

asr_pipeline = None
try:
    from peft import PeftModel
    from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline as _pipeline

    base_model = WhisperForConditionalGeneration.from_pretrained(ASR_BASE_MODEL)
    model_asr = PeftModel.from_pretrained(base_model, ASR_MODEL_PATH).to(DEVICE)
    processor_asr = WhisperProcessor.from_pretrained(ASR_BASE_MODEL)
    asr_pipeline = _pipeline(
        "automatic-speech-recognition", model=model_asr,
        tokenizer=processor_asr.tokenizer, feature_extractor=processor_asr.feature_extractor,
        device=0 if DEVICE == "cuda" else -1,
    )
    print("✅ ASR (LoRA) chargé pour le WER audio.")
except Exception as e:
    print(f"⚠️  ASR non chargé ({e}) — le WER audio sera ignoré (NaN).")

try:
    from jiwer import wer as jiwer_wer
    JIWER_OK = True
except ImportError:
    JIWER_OK = False


def normaliser_texte(texte):
    texte = str(texte).lower().strip()
    texte = unicodedata.normalize("NFKC", texte)
    texte = re.sub(r"[^\w\sàâäéèêëîïôöùûüÿñç]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte


wer_scores = []
for _, row in df_generation.iterrows():
    if row["audio_path"] is None or asr_pipeline is None or not JIWER_OK:
        wer_scores.append(None)
        continue
    try:
        texte_reconnu = asr_pipeline(row["audio_path"], generate_kwargs={"task": "transcribe"})["text"].strip()
        score = jiwer_wer(normaliser_texte(row["texte_wolof"]), normaliser_texte(texte_reconnu))
        wer_scores.append(round(score, 4))
    except Exception:
        wer_scores.append(None)

df_generation["wer_audio"] = wer_scores
display(df_generation[["phrase_id", "modele", "wer_audio", "snr_db", "rtf"]])

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

⚠️  ASR non chargé (Can't find 'adapter_config.json' at './wolof-whisper-small-lora') — le WER audio sera ignoré (NaN).


,phrase_id,modele,wer_audio,snr_db,rtf
0,1,SpeechT5_Bilal,None,39.87,0.264
1,1,SpeechT5_Bilal_v02,None,38.69,0.259
2,1,SpeechT5_AYI,None,43.10,0.250
3,2,SpeechT5_Bilal,None,43.41,0.258
4,2,SpeechT5_Bilal_v02,None,37.52,0.254
5,2,SpeechT5_AYI,None,43.48,0.248
6,3,SpeechT5_Bilal,None,40.76,0.251
7,3,SpeechT5_Bilal_v02,None,31.45,0.271
8,3,SpeechT5_AYI,None,40.14,0.246
9,4,SpeechT5_Bilal,None,40.08,0.234


## 8. Résumé par modèle et sélection du meilleur

In [15]:
colonnes_agg = [c for c in ["wer_audio", "snr_db", "rtf", "latence_ms"] if c in df_generation.columns]
df_resume = df_generation.groupby("modele")[colonnes_agg].mean(numeric_only=True).round(3)
df_resume["taux_echec"] = df_generation.groupby("modele")["echec"].mean().round(3)
df_resume.to_csv(RESULTS_DIR / "tts_resume.csv")

display(df_resume)


def normaliser(serie, inverser=False):
    s = serie.dropna()
    if s.empty or s.max() == s.min():
        return pd.Series(0.5, index=serie.index)
    norm = (serie - s.min()) / (s.max() - s.min())
    return 1 - norm if inverser else norm


ponderations = [("wer_audio", 0.40, True), ("snr_db", 0.25, False), ("rtf", 0.20, True), ("taux_echec", 0.15, True)]

score = pd.Series(0.0, index=df_resume.index)
poids_total = pd.Series(0.0, index=df_resume.index)
for col, poids, inverser in ponderations:
    if col not in df_resume.columns:
        continue
    dispo = df_resume[col].notna()
    if not dispo.any():
        continue
    score += normaliser(df_resume[col], inverser=inverser).fillna(0) * poids
    poids_total += dispo.astype(float) * poids

df_resume["score_composite"] = (score / poids_total.replace(0, np.nan)).round(4)
display(df_resume.sort_values("score_composite", ascending=False))

candidats = df_resume[df_resume.index.isin(tts_engines.keys()) & df_resume["score_composite"].notna()]
if not candidats.empty:
    nom_meilleur = candidats["score_composite"].idxmax()
    print(f"\n🏆 Meilleur modèle TTS Wolof : {nom_meilleur} (score = {candidats.loc[nom_meilleur, 'score_composite']:.3f})")
else:
    print("\n⚠️ Pas assez de métriques valides pour sélectionner automatiquement le meilleur modèle.")

,snr_db,rtf,latence_ms,taux_echec
modele,,,,
SpeechT5_AYI,41.392,0.260,620.70,0.0
SpeechT5_Bilal,41.574,0.276,611.22,0.0
SpeechT5_Bilal_v02,34.394,0.281,445.20,0.0


,snr_db,rtf,latence_ms,taux_echec,score_composite
modele,,,,,
SpeechT5_AYI,41.392,0.260,620.70,0.0,0.8644
SpeechT5_Bilal,41.574,0.276,611.22,0.0,0.6210
SpeechT5_Bilal_v02,34.394,0.281,445.20,0.0,0.1250



🏆 Meilleur modèle TTS Wolof : SpeechT5_AYI (score = 0.864)


## 9. Sauvegarde du meilleur modèle sur disque

In [17]:
# Pour le pusher sur HuggingFace Hub :
# Récupérer le meilleur modèle et son tokenizer
#hf_YOUR_TOKEN_HERE
modele = modeles[nom_meilleur]
tokenizer = tokenizers[nom_meilleur]

from huggingface_hub import notebook_login

notebook_login()

modele.push_to_hub("EvaF28/T-bot_TTS")
tokenizer.push_to_hub("EvaF28/T-bot_TTS")

NameError: name 'modeles' is not defined

## Bilan et limites

- Les WER/SNR/RTF donnent une base objective, mais **le MOS (naturel perçu à l'oreille)** reste la métrique la plus importante pour du TTS — écoute les audios de la Cellule 5 toi-même avant de trancher définitivement.
- `ADIA_TTS` et `Oolel_Voices` utilisent un chargement générique faute de documentation vérifiée — si l'un des deux échoue, va consulter sa fiche Hugging Face et adapte la fonction correspondante en Cellule 3.
- Jeu de test réduit à 5 phrases — élargis-le pour un résultat plus robuste, notamment en couvrant du vocabulaire médical et administratif propre à TontoumaBot.